# **DSA LAb Final Exam**
# **Team Member**
# **Saira Illyas**
# **B24F1000DS044**

In [1]:


from dataclasses import dataclass, asdict
from collections import deque
import json
import os

@dataclass
class Book:
    isbn: str
    title: str
    author: str
    year: int
    is_issued: bool = False
    issued_to: str = ""

BOOKS_FILE = "books.json"

books = []
isbn_index = {}
issue_queue = deque()   # each item: {"isbn": ..., "user_name": ...}

# ---------- Helpers ----------

def rebuild_index():
    global isbn_index
    isbn_index = {b.isbn: i for i, b in enumerate(books)}

def load_data():
    """Load books + queue from JSON file (if it exists)."""
    global books, issue_queue
    if not os.path.exists(BOOKS_FILE):
        print("No existing file found, starting with empty library.")
        books = []
        issue_queue = deque()
        rebuild_index()
        return

    with open(BOOKS_FILE, "r", encoding="utf-8") as f:
        data = json.load(f)

    books_raw = data.get("books", [])
    queue_raw = data.get("issue_queue", [])

    books = [Book(**b) for b in books_raw]
    issue_queue = deque(queue_raw)
    rebuild_index()
    print(f"Loaded {len(books)} books from {BOOKS_FILE}.")

def save_data():
    """Save books + queue to JSON file."""
    data = {
        "books": [asdict(b) for b in books],
        "issue_queue": list(issue_queue),
    }
    with open(BOOKS_FILE, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)
    print(f"Saved {len(books)} books to {BOOKS_FILE}.")

# ---------- Core operations ----------

def add_book(isbn, title, author, year):
    if isbn in isbn_index:
        print("Book with this ISBN already exists.")
        return
    b = Book(isbn=isbn, title=title, author=author, year=int(year))
    books.append(b)
    rebuild_index()
    save_data()
    print("Book added.")

def edit_book(isbn, new_title=None, new_author=None, new_year=None):
    idx = isbn_index.get(isbn)
    if idx is None:
        print("Book not found.")
        return
    b = books[idx]
    if new_title:
        b.title = new_title
    if new_author:
        b.author = new_author
    if new_year:
        b.year = int(new_year)
    save_data()
    print("Book updated.")

def delete_book(isbn):
    global books
    idx = isbn_index.get(isbn)
    if idx is None:
        print("Book not found.")
        return
    # remove from list
    removed = books.pop(idx)
    # remove requests for this book from queue
    global issue_queue
    issue_queue = deque([r for r in issue_queue if r["isbn"] != isbn])
    rebuild_index()
    save_data()
    print(f"Deleted: {removed.title}")

def search_by_isbn(isbn):
    idx = isbn_index.get(isbn)
    if idx is None:
        print("No book found with that ISBN.")
        return None
    b = books[idx]
    print_book(b)
    return b

def search_by_title(title):
    found = [b for b in books if b.title.lower() == title.lower()]
    if not found:
        print("No book found with that title.")
        return []
    for b in found:
        print_book(b)
    return found

def insertion_sort_by_title(book_list):
    """Return new list sorted by title using insertion sort (DSA requirement)."""
    arr = book_list[:]
    for i in range(1, len(arr)):
        key = arr[i]
        j = i - 1
        while j >= 0 and arr[j].title.lower() > key.title.lower():
            arr[j + 1] = arr[j]
            j -= 1
        arr[j + 1] = key
    return arr

def list_all_books():
    if not books:
        print("No books in library.")
        return
    sorted_books = insertion_sort_by_title(books)
    for b in sorted_books:
        print_book(b)

def list_available_books():
    available = [b for b in books if not b.is_issued]
    if not available:
        print("No available books.")
        return
    for b in insertion_sort_by_title(available):
        print_book(b)

def list_issued_books():
    issued = [b for b in books if b.is_issued]
    if not issued:
        print("No issued books.")
        return
    for b in issued:
        print_book(b)

def print_book(b: Book):
    print("-" * 40)
    print(f"ISBN     : {b.isbn}")
    print(f"Title    : {b.title}")
    print(f"Author   : {b.author}")
    print(f"Year     : {b.year}")
    print(f"Status   : {'Issued' if b.is_issued else 'Available'}")
    if b.is_issued:
        print(f"Issued to: {b.issued_to}")

# ---------- Issue / Return + Queue ----------

def direct_issue(isbn, user_name):
    idx = isbn_index.get(isbn)
    if idx is None:
        print("Book not found.")
        return
    b = books[idx]
    if b.is_issued:
        print(f"Book is already issued to {b.issued_to}.")
        return
    b.is_issued = True
    b.issued_to = user_name
    save_data()
    print(f"Book issued to {user_name}.")

def return_book(isbn):
    global issue_queue   # Correct place for global declaration

    idx = isbn_index.get(isbn)
    if idx is None:
        print("Book not found.")
        return

    b = books[idx]
    if not b.is_issued:
        print("Book is not issued.")
        return

    prev = b.issued_to
    b.is_issued = False
    b.issued_to = ""
    print(f"Book returned from {prev}.")

    # Auto-issue to next request in queue
    new_queue = deque()
    issued_to_next = False

    while issue_queue:
        req = issue_queue.popleft()

        if not issued_to_next and req["isbn"] == isbn:
            b.is_issued = True
            b.issued_to = req["user_name"]
            issued_to_next = True
            print(f"Automatically issued to next in queue: {req['user_name']}")
        else:
            new_queue.append(req)

    issue_queue = new_queue # This correctly updates the global issue_queue
    save_data() # Save once after all modifications

    # The following lines were the cause of the error and are redundant:
    # global issue_queue
    # issue_queue = new_queue
    # save_data()

def enqueue_issue_request(isbn, user_name):
    if isbn not in isbn_index:
        print("Book not found for this ISBN.")
        return
    issue_queue.append({"isbn": isbn, "user_name": user_name})
    save_data()
    print("Request added to queue.")

def process_next_request():
    if not issue_queue:
        print("No pending requests.")
        return
    req = issue_queue.popleft()
    isbn = req["isbn"]
    user_name = req["user_name"]
    idx = isbn_index.get(isbn)
    if idx is None:
        print("Book for this request does not exist anymore.")
        return
    b = books[idx]
    if not b.is_issued:
        b.is_issued = True
        b.issued_to = user_name
        print(f"Request processed. Issued to {user_name}.")
    else:
        print("Book still issued, request put back at end of queue.")
        issue_queue.append(req)
    save_data()

def show_queue():
    if not issue_queue:
        print("Queue is empty.")
        return
    print("Issue Queue (FIFO):")
    for i, req in enumerate(issue_queue, start=1):
        print(f"{i}. {req['user_name']} -> {req['isbn']}")

# ---------- Load data when cell runs ----------
load_data()

No existing file found, starting with empty library.


In [2]:
# Example usage (you can delete these later or keep as test):
add_book("111", "DSA Book", "Ali", 2023)
add_book("222", "Python Basics", "Sara", 2022)
list_all_books()
enqueue_issue_request("111", "B24F1000")
show_queue()
process_next_request()
list_all_books()


Saved 1 books to books.json.
Book added.
Saved 2 books to books.json.
Book added.
----------------------------------------
ISBN     : 111
Title    : DSA Book
Author   : Ali
Year     : 2023
Status   : Available
----------------------------------------
ISBN     : 222
Title    : Python Basics
Author   : Sara
Year     : 2022
Status   : Available
Saved 2 books to books.json.
Request added to queue.
Issue Queue (FIFO):
1. B24F1000 -> 111
Request processed. Issued to B24F1000.
Saved 2 books to books.json.
----------------------------------------
ISBN     : 111
Title    : DSA Book
Author   : Ali
Year     : 2023
Status   : Issued
Issued to: B24F1000
----------------------------------------
ISBN     : 222
Title    : Python Basics
Author   : Sara
Year     : 2022
Status   : Available


In [3]:
# === Library Management System - Python Backend (for Colab) ===
# DSA used:
#   - List of Book objects
#   - Dict (hash map) for ISBN -> index
#   - Queue (collections.deque) for issue requests (FIFO)
#   - Insertion sort for sorted display
# Persistence:
#   - JSON file books.json

from dataclasses import dataclass, asdict
from collections import deque
import json
import os

@dataclass
class Book:
    isbn: str
    title: str
    author: str
    year: int
    is_issued: bool = False
    issued_to: str = ""

BOOKS_FILE = "books.json"

books = []
isbn_index = {}
issue_queue = deque()   # each item: {"isbn": ..., "user_name": ...}

# ---------- Helpers ----------

def rebuild_index():
    global isbn_index
    isbn_index = {b.isbn: i for i, b in enumerate(books)}

def load_data():
    """Load books + queue from JSON file (if it exists)."""
    global books, issue_queue
    if not os.path.exists(BOOKS_FILE):
        print("No existing file found, starting with empty library.")
        books = []
        issue_queue = deque()
        rebuild_index()
        return

    with open(BOOKS_FILE, "r", encoding="utf-8") as f:
        data = json.load(f)

    books_raw = data.get("books", [])
    queue_raw = data.get("issue_queue", [])

    books = [Book(**b) for b in books_raw]
    issue_queue = deque(queue_raw)
    rebuild_index()
    print(f"Loaded {len(books)} books from {BOOKS_FILE}.")

def save_data():
    """Save books + queue to JSON file."""
    data = {
        "books": [asdict(b) for b in books],
        "issue_queue": list(issue_queue),
    }
    with open(BOOKS_FILE, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)
    print(f"Saved {len(books)} books to {BOOKS_FILE}.")

# ---------- Core operations ----------

def print_book(b: Book):
    print("-" * 40)
    print(f"ISBN     : {b.isbn}")
    print(f"Title    : {b.title}")
    print(f"Author   : {b.author}")
    print(f"Year     : {b.year}")
    print(f"Status   : {'Issued' if b.is_issued else 'Available'}")
    if b.is_issued:
        print(f"Issued to: {b.issued_to}")

def add_book(isbn, title, author, year):
    if isbn in isbn_index:
        print("Book with this ISBN already exists.")
        return
    b = Book(isbn=isbn, title=title, author=author, year=int(year))
    books.append(b)
    rebuild_index()
    save_data()
    print("Book added.")

def edit_book(isbn, new_title=None, new_author=None, new_year=None):
    idx = isbn_index.get(isbn)
    if idx is None:
        print("Book not found.")
        return
    b = books[idx]
    if new_title:
        b.title = new_title
    if new_author:
        b.author = new_author
    if new_year:
        b.year = int(new_year)
    save_data()
    print("Book updated.")

def delete_book(isbn):
    global issue_queue   # we reassign issue_queue, so declare global at top
    idx = isbn_index.get(isbn)
    if idx is None:
        print("Book not found.")
        return
    removed = books.pop(idx)
    # remove requests for this book from queue
    issue_queue = deque([r for r in issue_queue if r["isbn"] != isbn])
    rebuild_index()
    save_data()
    print(f"Deleted: {removed.title}")

def search_by_isbn(isbn):
    idx = isbn_index.get(isbn)
    if idx is None:
        print("No book found with that ISBN.")
        return None
    b = books[idx]
    print_book(b)
    return b

def search_by_title(title):
    found = [b for b in books if b.title.lower() == title.lower()]
    if not found:
        print("No book found with that title.")
        return []
    for b in found:
        print_book(b)
    return found

def insertion_sort_by_title(book_list):
    """Return new list sorted by title using insertion sort (DSA requirement)."""
    arr = book_list[:]
    for i in range(1, len(arr)):
        key = arr[i]
        j = i - 1
        while j >= 0 and arr[j].title.lower() > key.title.lower():
            arr[j + 1] = arr[j]
            j -= 1
        arr[j + 1] = key
    return arr

def list_all_books():
    if not books:
        print("No books in library.")
        return
    sorted_books = insertion_sort_by_title(books)
    for b in sorted_books:
        print_book(b)

def list_available_books():
    available = [b for b in books if not b.is_issued]
    if not available:
        print("No available books.")
        return
    for b in insertion_sort_by_title(available):
        print_book(b)

def list_issued_books():
    issued = [b for b in books if b.is_issued]
    if not issued:
        print("No issued books.")
        return
    for b in issued:
        print_book(b)

# ---------- Issue / Return + Queue ----------

def direct_issue(isbn, user_name):
    idx = isbn_index.get(isbn)
    if idx is None:
        print("Book not found.")
        return
    b = books[idx]
    if b.is_issued:
        print(f"Book is already issued to {b.issued_to}.")
        return
    b.is_issued = True
    b.issued_to = user_name
    save_data()
    print(f"Book issued to {user_name}.")

def return_book(isbn):
    global issue_queue   # declare global before using/reassigning

    idx = isbn_index.get(isbn)
    if idx is None:
        print("Book not found.")
        return

    b = books[idx]
    if not b.is_issued:
        print("Book is not issued.")
        return

    prev = b.issued_to
    b.is_issued = False
    b.issued_to = ""
    print(f"Book returned from {prev}.")

    # Auto-issue to next request in queue
    new_queue = deque()
    issued_to_next = False

    while issue_queue:
        req = issue_queue.popleft()
        if not issued_to_next and req["isbn"] == isbn:
            b.is_issued = True
            b.issued_to = req["user_name"]
            issued_to_next = True
            print(f"Automatically issued to next in queue: {req['user_name']}")
        else:
            new_queue.append(req)

    issue_queue = new_queue
    save_data()

def enqueue_issue_request(isbn, user_name):
    if isbn not in isbn_index:
        print("Book not found for this ISBN.")
        return
    issue_queue.append({"isbn": isbn, "user_name": user_name})
    save_data()
    print("Request added to queue.")

def process_next_request():
    if not issue_queue:
        print("No pending requests.")
        return
    req = issue_queue.popleft()
    isbn = req["isbn"]
    user_name = req["user_name"]
    idx = isbn_index.get(isbn)
    if idx is None:
        print("Book for this request does not exist anymore.")
        save_data()
        return
    b = books[idx]
    if not b.is_issued:
        b.is_issued = True
        b.issued_to = user_name
        print(f"Request processed. Issued to {user_name}.")
    else:
        print("Book still issued, request put back at end of queue.")
        issue_queue.append(req)
    save_data()

def show_queue():
    if not issue_queue:
        print("Queue is empty.")
        return
    print("Issue Queue (FIFO):")
    for i, req in enumerate(issue_queue, start=1):
        print(f"{i}. {req['user_name']} -> {req['isbn']}")

# ---------- Load data when cell runs ----------
load_data()


Loaded 2 books from books.json.


In [4]:
# Quick test (optional)
add_book("111", "DSA Book", "Ali", 2023)
add_book("222", "Python Basics", "Sara", 2022)
list_all_books()
enqueue_issue_request("111", "B24F1000")
show_queue()
process_next_request()
list_all_books()


Book with this ISBN already exists.
Book with this ISBN already exists.
----------------------------------------
ISBN     : 111
Title    : DSA Book
Author   : Ali
Year     : 2023
Status   : Issued
Issued to: B24F1000
----------------------------------------
ISBN     : 222
Title    : Python Basics
Author   : Sara
Year     : 2022
Status   : Available
Saved 2 books to books.json.
Request added to queue.
Issue Queue (FIFO):
1. B24F1000 -> 111
Book still issued, request put back at end of queue.
Saved 2 books to books.json.
----------------------------------------
ISBN     : 111
Title    : DSA Book
Author   : Ali
Year     : 2023
Status   : Issued
Issued to: B24F1000
----------------------------------------
ISBN     : 222
Title    : Python Basics
Author   : Sara
Year     : 2022
Status   : Available


In [5]:
%%writefile index.html
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <title>Library Management System - Frontend Prototype</title>
  <meta name="viewport" content="width=device-width, initial-scale=1" />
  <style>
    body {
      margin: 0;
      font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
      background: #020617;
      color: #e5e7eb;
    }
    .container {
      max-width: 1000px;
      margin: 0 auto;
      padding: 20px 14px 40px;
    }
    h1 {
      font-size: 1.8rem;
      margin-bottom: 4px;
    }
    .subtitle {
      color: #9ca3af;
      font-size: 0.9rem;
      margin-bottom: 16px;
    }
    .grid {
      display: grid;
      grid-template-columns: 1.1fr 1.4fr;
      gap: 12px;
    }
    @media (max-width: 800px) {
      .grid { grid-template-columns: 1fr; }
    }
    .card {
      background: #020617;
      border-radius: 12px;
      border: 1px solid #1f2937;
      padding: 12px 12px 14px;
      box-shadow: 0 14px 35px rgba(15,23,42,0.85);
    }
    .card h2 {
      font-size: 1.05rem;
      margin: 0 0 8px;
    }
    label {
      display: block;
      font-size: 0.8rem;
      color: #9ca3af;
      margin-bottom: 3px;
    }
    input {
      width: 100%;
      padding: 7px 9px;
      border-radius: 8px;
      border: 1px solid #1f2937;
      background: #020617;
      color: #e5e7eb;
      font-size: 0.85rem;
      margin-bottom: 8px;
      box-sizing: border-box;
    }
    input:focus {
      outline: none;
      border-color: #22c55e;
      box-shadow: 0 0 0 1px rgba(34,197,94,0.4);
    }
    button {
      border-radius: 999px;
      border: none;
      padding: 7px 14px;
      font-size: 0.84rem;
      cursor: pointer;
      margin-right: 6px;
      margin-top: 2px;
    }
    .btn-primary {
      background: linear-gradient(135deg, #22c55e, #4ade80);
      color: #022c22;
    }
    .btn-outline {
      background: transparent;
      color: #9ca3af;
      border: 1px solid #4b5563;
    }
    .btn-danger {
      background: #ef4444;
      color: #fee2e2;
    }
    table {
      width: 100%;
      border-collapse: collapse;
      font-size: 0.8rem;
    }
    th, td {
      border-bottom: 1px solid #0b1120;
      padding: 7px 8px;
      text-align: left;
    }
    th {
      color: #9ca3af;
      position: sticky;
      top: 0;
      background: #020617;
    }
    tbody tr:nth-child(even) {
      background: rgba(15,23,42,0.4);
    }
    tbody tr:hover {
      background: rgba(15,23,42,0.9);
    }
    .status-pill {
      display: inline-block;
      padding: 2px 7px;
      border-radius: 999px;
      font-size: 0.7rem;
    }
    .status-available {
      background: rgba(34,197,94,0.12);
      color: #4ade80;
      border: 1px solid rgba(34,197,94,0.5);
    }
    .status-issued {
      background: rgba(239,68,68,0.12);
      color: #fecaca;
      border: 1px solid rgba(239,68,68,0.6);
    }
    .hint {
      font-size: 0.76rem;
      color: #9ca3af;
      margin-top: 4px;
    }
  </style>
</head>
<body>
  <div class="container">
    <h1>Library Management System</h1>
    <div class="subtitle">
      Frontend prototype (HTML + JS). Data is stored in browser localStorage.<br>
      For DSA + file-handling backend, see Python code in Colab.
    </div>

    <div class="grid">
      <!-- LEFT: FORM + FILTER -->
      <div class="card">
        <h2>Book Management</h2>
        <label>ISBN</label>
        <input id="isbn" placeholder="e.g. 978-1234567890" />
        <label>Title</label>
        <input id="title" placeholder="Book Title" />
        <label>Author</label>
        <input id="author" placeholder="Author Name" />
        <label>Year</label>
        <input id="year" type="number" placeholder="2024" />

        <button class="btn-primary" onclick="addOrUpdateBook()">Add / Update</button>
        <button class="btn-outline" onclick="resetForm()">Reset</button>

        <div class="hint">
          ISBN is treated as a unique key. If a book with the same ISBN exists, it will be updated.
        </div>

        <hr style="border-color:#111827; margin:10px 0;" />

        <h2>Filter</h2>
        <button class="btn-outline" onclick="setFilter('all')">All</button>
        <button class="btn-outline" onclick="setFilter('available')">Available</button>
        <button class="btn-outline" onclick="setFilter('issued')">Issued</button>

        <div class="hint">
          Sorted by title using insertion sort (implemented in JavaScript).
        </div>
      </div>

      <!-- RIGHT: TABLE -->
      <div class="card">
        <h2>Books List</h2>
        <div style="max-height:400px; overflow:auto; border:1px solid #111827; border-radius:8px;">
          <table>
            <thead>
              <tr>
                <th>ISBN</th>
                <th>Title</th>
                <th>Author</th>
                <th>Year</th>
                <th>Status</th>
                <th>Actions</th>
              </tr>
            </thead>
            <tbody id="booksBody">
            </tbody>
          </table>
        </div>
        <div id="noBooksMsg" class="hint"></div>
      </div>
    </div>
  </div>

  <script>
    // ============================
    // Library frontend logic (JS)
    // ============================
    const STORAGE_KEY = "frontend_books";
    let books = []; // each {isbn, title, author, year, isIssued, issuedTo}
    let filterMode = "all";

    function loadFromStorage() {
      try {
        const raw = localStorage.getItem(STORAGE_KEY);
        if (!raw) {
          books = [];
        } else {
          books = JSON.parse(raw);
        }
      } catch (e) {
        books = [];
      }
    }

    function saveToStorage() {
      localStorage.setItem(STORAGE_KEY, JSON.stringify(books));
    }

    function resetForm() {
      document.getElementById("isbn").value = "";
      document.getElementById("title").value = "";
      document.getElementById("author").value = "";
      document.getElementById("year").value = "";
    }

    function addOrUpdateBook() {
      const isbn = document.getElementById("isbn").value.trim();
      const title = document.getElementById("title").value.trim();
      const author = document.getElementById("author").value.trim();
      const year = document.getElementById("year").value.trim();

      if (!isbn || !title || !author || !year) {
        alert("Fill all fields.");
        return;
      }

      const existingIndex = books.findIndex(b => b.isbn === isbn);
      if (existingIndex !== -1) {
        // update
        books[existingIndex].title = title;
        books[existingIndex].author = author;
        books[existingIndex].year = parseInt(year, 10);
        alert("Book updated.");
      } else {
        books.push({
          isbn,
          title,
          author,
          year: parseInt(year, 10),
          isIssued: false,
          issuedTo: ""
        });
        alert("Book added.");
      }
      saveToStorage();
      renderBooks();
      resetForm();
    }

    function deleteBook(isbn) {
      if (!confirm("Delete this book?")) return;
      books = books.filter(b => b.isbn !== isbn);
      saveToStorage();
      renderBooks();
    }

    function issueOrReturn(isbn) {
      const idx = books.findIndex(b => b.isbn === isbn);
      if (idx === -1) return;
      const b = books[idx];
      if (!b.isIssued) {
        const user = prompt("Enter user name / ID:");
        if (!user) return;
        b.isIssued = true;
        b.issuedTo = user;
      } else {
        b.isIssued = false;
        b.issuedTo = "";
      }
      saveToStorage();
      renderBooks();
    }

    function setFilter(mode) {
      filterMode = mode;
      renderBooks();
    }

    // insertion sort by title
    function sortByTitle(arr) {
      const a = arr.slice();
      for (let i = 1; i < a.length; i++) {
        const key = a[i];
        let j = i - 1;
        while (j >= 0 && a[j].title.toLowerCase() > key.title.toLowerCase()) {
          a[j + 1] = a[j];
          j--;
        }
        a[j + 1] = key;
      }
      return a;
    }

    function renderBooks() {
      const tbody = document.getElementById("booksBody");
      const msg = document.getElementById("noBooksMsg");
      tbody.innerHTML = "";

      let filtered = books;
      if (filterMode === "available") {
        filtered = books.filter(b => !b.isIssued);
      } else if (filterMode === "issued") {
        filtered = books.filter(b => b.isIssued);
      }

      const sorted = sortByTitle(filtered);
      if (sorted.length === 0) {
        msg.textContent = "No books to display.";
        return;
      }
      msg.textContent = "";

      sorted.forEach(b => {
        const tr = document.createElement("tr");

        const tdIsbn = document.createElement("td");
        tdIsbn.textContent = b.isbn;

        const tdTitle = document.createElement("td");
        tdTitle.textContent = b.title;

        const tdAuthor = document.createElement("td");
        tdAuthor.textContent = b.author;

        const tdYear = document.createElement("td");
        tdYear.textContent = b.year;

        const tdStatus = document.createElement("td");
        const span = document.createElement("span");
        span.classList.add("status-pill");
        if (!b.isIssued) {
          span.classList.add("status-available");
          span.textContent = "Available";
        } else {
          span.classList.add("status-issued");
          span.textContent = "Issued (" + b.issuedTo + ")";
        }
        tdStatus.appendChild(span);

        const tdActions = document.createElement("td");
        const btnDel = document.createElement("button");
        btnDel.className = "btn-danger";
        btnDel.textContent = "Delete";
        btnDel.onclick = () => deleteBook(b.isbn);

        const btnIssue = document.createElement("button");
        btnIssue.className = "btn-primary";
        btnIssue.style.marginLeft = "4px";
        btnIssue.textContent = b.isIssued ? "Return" : "Issue";
        btnIssue.onclick = () => issueOrReturn(b.isbn);

        tdActions.appendChild(btnDel);
        tdActions.appendChild(btnIssue);

        tr.appendChild(tdIsbn);
        tr.appendChild(tdTitle);
        tr.appendChild(tdAuthor);
        tr.appendChild(tdYear);
        tr.appendChild(tdStatus);
        tr.appendChild(tdActions);

        tbody.appendChild(tr);
      });
    }

    // init
    loadFromStorage();
    renderBooks();
  </script>
</body>
</html>


Writing index.html
